In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted!")
print("\nYour files are at: /content/drive/MyDrive/mimic_capstone/")

Mounted at /content/drive
✅ Google Drive mounted!

Your files are at: /content/drive/MyDrive/mimic_capstone/


In [ ]:
!pip install transformers datasets accelerate sentencepiece -q

print("✅ Packages installed!")

✅ Packages installed!


In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print("✅ GPU READY! Training will be FAST!")
else:
    print("⚠️ No GPU - check Runtime settings")


CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
Memory: 39.6 GB
✅ GPU READY! Training will be FAST!


In [ ]:
import os

base_dir = '/content/drive/MyDrive/mimic_capstone'

# Data paths
data_dir = f'{base_dir}/data'
models_dir = f'{base_dir}/models'
results_dir = f'{base_dir}/results'

# Create directories if they don't exist
os.makedirs(models_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

print("✅ Paths configured!")
print(f"Data: {data_dir}")
print(f"Models: {models_dir}")
print(f"Results: {results_dir}")

✅ Paths configured!
Data: /content/drive/MyDrive/mimic_capstone/data
Models: /content/drive/MyDrive/mimic_capstone/models
Results: /content/drive/MyDrive/mimic_capstone/results


In [ ]:

import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from tqdm.auto import tqdm
import time

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)

print("🚀 Starting Agent 2 BERT Training")
print("=" * 60)

# =====================================================
# 1. LOAD DATA
# =====================================================
print("\n📂 Loading data...")
train_data = pd.read_csv(f'{data_dir}/notes_train_labeled.csv')
notes_val = pd.read_csv(f'{data_dir}/notes_val.csv')
labels = pd.read_csv(f'{data_dir}/patient_multilabels.csv')
val_data = notes_val.merge(labels, on='SUBJECT_ID', how='inner')

print(f"✅ Data loaded!")
print(f"   Training: {len(train_data):,} patients")
print(f"   Validation: {len(val_data):,} patients")

# Disease columns
disease_cols = [
    'SEPSIS', 'PNEUMONIA', 'RESPIRATORY_FAILURE',
    'ACUTE_KIDNEY_INJURY', 'HEART_FAILURE',
    'ATRIAL_FIBRILLATION', 'CORONARY_ARTERY_DISEASE',
    'ANEMIA', 'PANCREATITIS'
]

# =====================================================
# 2. LOAD TOKENIZER
# =====================================================
print("\n🔤 Loading Bio_ClinicalBERT tokenizer...")
model_name = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)
print(f"✅ Tokenizer loaded!")

# =====================================================
# 3. TOKENIZE DATA
# =====================================================
print("\n🔤 Tokenizing data (takes 2-3 minutes)...")

def tokenize_texts(texts, tokenizer, max_length=512):
    return tokenizer(
        texts.tolist(),
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_tensors='pt'
    )

train_encodings = tokenize_texts(train_data['CLINICAL_TEXT'], tokenizer)
val_encodings = tokenize_texts(val_data['CLINICAL_TEXT'], tokenizer)

print(f"✅ Tokenization complete!")

# =====================================================
# 4. TRAINING CONFIGURATION (GPU)
# =====================================================
training_args = TrainingArguments(
    output_dir=f'{models_dir}/agent2_checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    save_total_limit=1,
    fp16=True,  # Mixed precision for GPU
    report_to='none'
)

# =====================================================
# 5. METRICS FUNCTION
# =====================================================
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    probs = torch.softmax(torch.tensor(pred.predictions), dim=-1)[:, 1].numpy()

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)

    if len(np.unique(labels)) > 1:
        auc = roc_auc_score(labels, probs)
    else:
        auc = 0.0

    return {'accuracy': acc, 'f1': f1, 'auc': auc}

# =====================================================
# 6. TRAIN ALL MODELS
# =====================================================
print("\n🤖 Training 9 BERT models...")
print("⏰ GPU Training: ~10-15 min per disease (~2 hours total)")
print("=" * 60)

start_time = time.time()
val_metrics = {}

for idx, disease in enumerate(disease_cols, 1):
    disease_start = time.time()

    print(f"\n{'='*60}")
    print(f"🎯 Disease {idx}/9: {disease}")
    print(f"{'='*60}")

    # Get labels
    y_train = train_data[disease].values
    y_val = val_data[disease].values

    # Create datasets
    train_dataset = Dataset.from_dict({
        'input_ids': train_encodings['input_ids'],
        'attention_mask': train_encodings['attention_mask'],
        'labels': y_train
    })

    val_dataset = Dataset.from_dict({
        'input_ids': val_encodings['input_ids'],
        'attention_mask': val_encodings['attention_mask'],
        'labels': y_val
    })

    train_dataset.set_format('torch')
    val_dataset.set_format('torch')

    # Load model
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        problem_type="single_label_classification"
    )

    # Create trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    # Train
    print(f"\n🏋️ Training...")
    trainer.train()

    # Evaluate
    print(f"\n📊 Evaluating...")
    predictions = trainer.predict(val_dataset)
    y_pred_proba = torch.softmax(torch.tensor(predictions.predictions), dim=-1)[:, 1].numpy()
    y_pred = (y_pred_proba >= 0.5).astype(int)

    # Metrics
    accuracy = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_pred_proba)

    val_metrics[disease] = {
        'accuracy': accuracy,
        'f1_score': f1,
        'auc_roc': auc
    }

    # Time
    disease_time = (time.time() - disease_start) / 60
    elapsed = (time.time() - start_time) / 60
    remaining = disease_time * (len(disease_cols) - idx)

    print(f"\n✅ {disease} complete!")
    print(f"   Acc: {accuracy:.3f}, F1: {f1:.3f}, AUC: {auc:.3f}")
    print(f"   Time: {disease_time:.1f} min")
    print(f"   Progress: {idx}/{len(disease_cols)} ({elapsed:.1f} min elapsed, ~{remaining:.1f} min remaining)")

    # Save model
    model_path = f'{models_dir}/agent2_{disease.lower()}'
    model.save_pretrained(model_path)

    # Clean up GPU
    del model
    del trainer
    torch.cuda.empty_cache()

# =====================================================
# 7. SAVE RESULTS
# =====================================================
print(f"\n{'='*60}")
print(f"🎉 ALL MODELS TRAINED!")
print(f"⏰ Total time: {(time.time() - start_time) / 60:.1f} min")
print(f"{'='*60}")

# Save metrics
metrics_df = pd.DataFrame(val_metrics).T
metrics_df.to_csv(f'{results_dir}/agent2_val_metrics.csv')

print("\n📊 Final Results:")
print(metrics_df.round(3).to_string())

print(f"\n📈 Summary:")
print(f"   Average AUC: {metrics_df['auc_roc'].mean():.3f}")
print(f"   Average F1:  {metrics_df['f1_score'].mean():.3f}")

# Save tokenizer
tokenizer.save_pretrained(f'{models_dir}/agent2_tokenizer')

print(f"\n✅ All files saved to Google Drive!")
print(f"   Models: {models_dir}/agent2_*")
print(f"   Results: {results_dir}/agent2_val_metrics.csv")

🚀 Starting Agent 2 BERT Training

📂 Loading data...
✅ Data loaded!
   Training: 21,678 patients
   Validation: 4,643 patients

🔤 Loading Bio_ClinicalBERT tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

✅ Tokenizer loaded!

🔤 Tokenizing data (takes 2-3 minutes)...
✅ Tokenization complete!

🤖 Training 9 BERT models...
⏰ GPU Training: ~10-15 min per disease (~2 hours total)

🎯 Disease 1/9: SEPSIS


...

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



🏋️ Training...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Auc
1,0.354000,0.363349,0.864527,0.230110,0.736024
2,0.350400,0.363274,0.866681,0.231056,0.746339
3,0.348200,0.374283,0.859143,0.301282,0.746814



📊 Evaluating...



✅ PANCREATITIS complete!
   Acc: 0.859, F1: 0.301, AUC: 0.747
   Time: 4.5 min
   Progress: 9/9 (41.4 min elapsed, ~0.0 min remaining)

🎉 ALL MODELS TRAINED!
⏰ Total time: 41.4 min

📊 Final Results:
                         accuracy  f1_score  auc_roc
SEPSIS                      0.824     0.362    0.757
PNEUMONIA                   0.799     0.357    0.725
RESPIRATORY_FAILURE         0.694     0.501    0.728
ACUTE_KIDNEY_INJURY         0.702     0.561    0.744
HEART_FAILURE               0.760     0.616    0.802
ATRIAL_FIBRILLATION         0.781     0.630    0.823
CORONARY_ARTERY_DISEASE     0.816     0.736    0.873
ANEMIA                      0.673     0.323    0.655
PANCREATITIS                0.859     0.301    0.747

📈 Summary:
   Average AUC: 0.762
   Average F1:  0.487

✅ All files saved to Google Drive!
   Models: /content/drive/MyDrive/mimic_capstone/models/agent2_*
   Results: /content/drive/MyDrive/mimic_capstone/results/agent2_val_metrics.csv


In [ ]:


import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
import os

print("🎓 MS Capstone - Agent 2 Prediction Generation")
print("="*60)

# ============================================================
# 1. MOUNT DRIVE AND SET PATHS
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

base_dir = '/content/drive/MyDrive/mimic_capstone'
data_dir = f'{base_dir}/data'
models_dir = f'{base_dir}/models'
results_dir = f'{base_dir}/results'

print(f"\n✅ Paths set:")
print(f"   Models: {models_dir}")
print(f"   Data: {data_dir}")

# Create results directory if needed
os.makedirs(results_dir, exist_ok=True)

# ============================================================
# 2. VERIFY MODELS EXIST
# ============================================================
print("\n📂 Checking for trained models...")

diseases = [
    'sepsis', 'pneumonia', 'respiratory_failure',
    'acute_kidney_injury', 'heart_failure',
    'atrial_fibrillation', 'coronary_artery_disease',
    'anemia', 'pancreatitis'
]

DISEASE_MAP = {
    'sepsis': 'SEPSIS',
    'pneumonia': 'PNEUMONIA',
    'respiratory_failure': 'RESPIRATORY_FAILURE',
    'acute_kidney_injury': 'ACUTE_KIDNEY_INJURY',
    'heart_failure': 'HEART_FAILURE',
    'atrial_fibrillation': 'ATRIAL_FIBRILLATION',
    'coronary_artery_disease': 'CORONARY_ARTERY_DISEASE',
    'anemia': 'ANEMIA',
    'pancreatitis': 'PANCREATITIS'
}

available_models = []
for disease in diseases:
    model_path = f'{models_dir}/agent2_{disease}'
    if os.path.exists(model_path):
        print(f"   ✅ {disease}")
        available_models.append(disease)
    else:
        print(f"   ❌ {disease} - NOT FOUND")

if not available_models:
    raise FileNotFoundError("No Agent 2 models found! Please check the models directory.")

print(f"\n✅ Found {len(available_models)}/9 models")

# ============================================================
# 3. LOAD VALIDATION DATA
# ============================================================
print("\n📊 Loading validation data...")

# Load validation notes
val_file = f'{data_dir}/notes_val.csv'
if not os.path.exists(val_file):
    raise FileNotFoundError(f"Validation file not found: {val_file}")

notes_val = pd.read_csv(val_file)
print(f"   Loaded notes: {notes_val.shape}")

# Find text column
text_col = None
for col in ['CLINICAL_TEXT', 'TEXT', 'text', 'clinical_text', 'notes']:
    if col in notes_val.columns:
        text_col = col
        break

if text_col is None:
    print(f"   Available columns: {list(notes_val.columns)}")
    raise ValueError("Cannot find text column in validation data")

print(f"   Using text column: {text_col}")
print(f"   Validation patients: {len(notes_val)}")

# ============================================================
# 4. LOAD TOKENIZER
# ============================================================
print("\n🔤 Loading tokenizer...")

tokenizer_path = f'{models_dir}/agent2_tokenizer'
if os.path.exists(tokenizer_path):
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
    print(f"   ✅ Loaded from: {tokenizer_path}")
else:
    tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
    print(f"   ✅ Loaded from HuggingFace")

# ============================================================
# 5. TOKENIZE DATA
# ============================================================
print("\n🔤 Tokenizing validation data...")

encodings = tokenizer(
    notes_val[text_col].tolist(),
    truncation=True,
    padding='max_length',
    max_length=512,
    return_tensors='pt'
)

print(f"   ✅ Tokenized {len(notes_val)} patients")

# Create dataset
input_ids = encodings['input_ids']
attention_mask = encodings['attention_mask']

dataset = TensorDataset(input_ids, attention_mask)
dataloader = DataLoader(dataset, batch_size=16, shuffle=False)

# ============================================================
# 6. GENERATE PREDICTIONS
# ============================================================
print("\n🤖 Generating predictions from trained models...")
print("   ⏰ Estimated time: 3-5 minutes")
print("="*60)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\n   Device: {device}")

all_predictions = {}

for disease in tqdm(available_models, desc="Diseases"):
    # Load model
    model_path = f'{models_dir}/agent2_{disease}'
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    model.to(device)
    model.eval()

    # Collect predictions
    predictions = []

    # Run inference
    with torch.no_grad():
        for batch_input_ids, batch_attention_mask in tqdm(dataloader,
                                                           desc=f"  {disease}",
                                                           leave=False):
            # Move to device
            batch_input_ids = batch_input_ids.to(device)
            batch_attention_mask = batch_attention_mask.to(device)

            # Forward pass
            outputs = model(
                input_ids=batch_input_ids,
                attention_mask=batch_attention_mask
            )

            # Get probabilities for positive class
            logits = outputs.logits
            probs = torch.softmax(logits, dim=1)[:, 1]

            predictions.extend(probs.cpu().numpy())

    # Store predictions
    all_predictions[DISEASE_MAP[disease]] = np.array(predictions)

    # Clean up
    del model
    if device == 'cuda':
        torch.cuda.empty_cache()

print(f"\n✅ Generated predictions for {len(all_predictions)} diseases")

# ============================================================
# 7. CREATE PREDICTIONS DATAFRAME
# ============================================================
print("\n📋 Creating predictions file...")

predictions_df = pd.DataFrame({
    'SUBJECT_ID': notes_val['SUBJECT_ID'].values
})

for disease, preds in all_predictions.items():
    predictions_df[disease] = preds

print(f"\n   Shape: {predictions_df.shape}")
print(f"   Columns: {list(predictions_df.columns)}")

# Show sample
print(f"\n   Sample predictions:")
print(predictions_df.head(3).round(3))

# Show statistics
print(f"\n   Prediction statistics:")
for disease in predictions_df.columns[1:]:  # Skip SUBJECT_ID
    mean_prob = predictions_df[disease].mean()
    min_prob = predictions_df[disease].min()
    max_prob = predictions_df[disease].max()
    print(f"   {disease:30s} Mean: {mean_prob:.3f}  Range: [{min_prob:.3f}, {max_prob:.3f}]")

# ============================================================
# 8. SAVE AND DOWNLOAD
# ============================================================
print("\n💾 Saving predictions...")

# Save to Drive
output_path = f'{results_dir}/agent2_validation_predictions.csv'
predictions_df.to_csv(output_path, index=False)

print(f"   ✅ Saved to Drive: {output_path}")

# Download to local computer
print("\n📥 Downloading to your computer...")
from google.colab import files
files.download(output_path)

print("\n" + "="*60)
print("🎉 SUCCESS!")
print("="*60)
print("\n✅ Generated REAL predictions from trained BERT models")
print(f"   Patients: {len(predictions_df)}")
print(f"   Diseases: {len(all_predictions)}")
print(f"   File: agent2_validation_predictions.csv")
print("\n📂 Next steps:")
print("   1. Save the downloaded file")
print("   2. Place in: mimic_project/results/")
print("   3. Continue with fusion training")
print("="*60)